# Voxtral-Mini-3B on AWS Neuron (trn2.3xlarge)

This notebook demonstrates how to download, compile, and serve
**Voxtral-Mini-3B** (`mistralai/Voxtral-Mini-3B-2507`) on `trn2.3xlarge`
using **vLLM-neuron** with the NxDI (`neuronx-distributed-inference`)
Voxtral contrib.

## Instance Setup

This notebook is designed to run on a **trn2.3xlarge** with the
**Deep Learning AMI Neuron (Ubuntu 24.04) 20260522** (Neuron SDK 2.30).

To launch your instance, use the AWS Console or CLI.  For a walkthrough
of launching a Neuron instance, see this video tutorial (starts at the
instance launch section):

> **Video Guide**: [Launching a Neuron Instance](https://youtu.be/CyTCTuq1z0Q?t=657)

## Jupyter Kernel Setup

This notebook requires a Python kernel from the pre-installed Neuron
virtual environment.  There are two ways to set this up:

### Option A: Jupyter Server on the Instance

SSH into your instance and start Jupyter from the Neuron virtual
environment:

```bash
source /opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/bin/activate
pip install jupyter
jupyter notebook --no-browser --port=8888
```

Then use SSH port forwarding to access it from your local browser:

```bash
ssh -i "/path/to/sshkey.pem" -L 8888:localhost:8888 ubuntu@<instance_ip>
```

### Option B: VS Code Remote-SSH

With Visual Studio Code installed on your local machine, you can use
Remote-SSH to edit and run notebooks directly on the Neuron instance:

1. Select **Remote-SSH: Connect to Host...** from the Command Palette
   (`F1` or `Shift+Cmd+P`)
2. Enter the full connection string:
   `ssh -i "/path/to/sshkey.pem" ubuntu@<instance_ip>`
3. VS Code will connect and set up the VS Code server automatically
4. When prompted, browse to your working directory on the instance
5. Some menu commands may appear greyed out, but keyboard shortcuts
   still work (`Cmd+S` to save, `` Ctrl+Shift+` `` for terminal).  You
   may need to restart VS Code.

To use the pre-installed Neuron virtual environment as your Jupyter
kernel in VS Code, open a terminal and create a symbolic link:

```bash
ln -s /opt/aws_neuronx_venv_pytorch_inference_vllm_0_16 ~/.venv
```

Then select the `.venv` Python interpreter when choosing a kernel for
the notebook.

## What this notebook does

1. Clone the `jimburtoft` forks of `neuronx-distributed-inference`
   (branch `contrib/voxtral-mini-3B`) and `vllm-neuron` (branch
   `contrib/voxtral-0.5.0`).  These add Voxtral support that is not yet
   merged into the upstream projects.
2. Install both forks in the pre-installed SDK 2.30 vLLM venv.
3. Download the Voxtral-Mini-3B checkpoint from HuggingFace.
4. Compile the model to Neuron (one-time; a few minutes).
5. Run a single-file smoke transcription via vLLM.
6. Run the benchmark harness across your own audio dataset and report
   per-file mean latency.

The performance target is **mean ≤ 543 ms per file** on a mix of 0-30 s
speech clips at TP=4, LNC=2, bfloat16, SDK 2.30.


## 1. Verify the instance

Run `neuron-ls` and check that you have four logical NeuronCores
(LNC=2 default on trn2.3xlarge).


In [1]:
!neuron-ls


instance-type: trn2.3xlarge
instance-id: i-03e610b910fe60156
logical-neuroncore-config: 2
+--------+--------+----------+--------+--------------+----------+------+
| NEURON | NEURON |  NEURON  | NEURON |     PCI      |   CPU    | NUMA |
| DEVICE | CORES  | CORE IDS | MEMORY |     BDF      | AFFINITY | NODE |
+--------+--------+----------+--------+--------------+----------+------+
| 0      | 4      | 0-3      | 96 GB  | 0000:33:00.0 | 0-11     | 0    |
+--------+--------+----------+--------+--------------+----------+------+


## 2. Activate the pre-installed venv

The SDK 2.30 DLAMI ships two venvs relevant to this notebook:

- `/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/` -- NxDI only.
- `/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/` -- vLLM + NxDI.
  **This is the one we want.**

We assume your Jupyter kernel already comes from the vLLM venv (see the
"Jupyter Kernel Setup" section at the top of the notebook).  Verify
that the runtime venv is correct:


In [2]:
import sys
print("Python:", sys.executable)
assert "aws_neuronx_venv_pytorch_inference_vllm_0_16" in sys.executable, (
    "This notebook expects the pre-installed vLLM venv.  See kernel setup."
)


Python: /opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/bin/python3


## 3. Clone the forks

Both the Voxtral NxDI implementation and the vLLM-neuron Voxtral
integration live on `jimburtoft`-hosted forks.  Clone them into your
home directory.  The NxDI branch will be added to `PYTHONPATH` at
runtime; the vLLM-neuron branch is installed via `pip install -e`.


In [3]:
import os
from pathlib import Path

HOME = Path(os.path.expanduser("~"))
NXDI_FORK = HOME / "neuronx-distributed-inference"
VLLM_FORK = HOME / "vllm-neuron"

if not NXDI_FORK.exists():
    !git clone -b contrib/voxtral-mini-3B \
        https://github.com/jimburtoft/neuronx-distributed-inference.git \
        {NXDI_FORK}
else:
    print(f"{NXDI_FORK} already exists, skipping clone.")

if not VLLM_FORK.exists():
    !git clone -b contrib/voxtral-0.5.0 \
        https://github.com/jimburtoft/vllm-neuron.git \
        {VLLM_FORK}
else:
    print(f"{VLLM_FORK} already exists, skipping clone.")


Cloning into '/home/ubuntu/neuronx-distributed-inference'...


remote: Enumerating objects: 10408, done.


remote: Counting objects: 100% (244/244), done.
remote: Compressing objects: 100% (111/111), done.


remote: Total 10408 (delta 168), reused 179 (delta 131), pack-reused 10164 (from 2)
Receiving objects: 100% (10408/10408), 21.53 MiB | 20.92 MiB/s, done.


Resolving deltas: 100% (4752/4752), done.


Cloning into '/home/ubuntu/vllm-neuron'...


remote: Enumerating objects: 884, done.
remote: Counting ob

remote: Counting objects: 100% (259/259), done.


remote: Compressing objects: 100% (103/103), done.


remote: Total 884 (delta 199), reused 156 (delta 156), pack-reused 625 (from 2)
Receiving objects: 100% (884/884), 649.48 KiB | 6.43 MiB/s, done.
Resolving deltas: 100% (421/421), done.


## 4. Install the vLLM-neuron fork

This replaces the DLAMI's pre-installed `vllm-neuron` package with the
patched fork.  The install is editable (`-e`) so future `git pull`
updates take effect without a reinstall.


In [4]:
!pip install -e {VLLM_FORK} 2>&1 | tail -5


  Attempting uninstall: vllm-neuron
    Found existing installation: vllm-neuron 0.5.0
    Uninstalling vllm-neuron-0.5.0:
      Successfully uninstalled vllm-neuron-0.5.0


Verify the install picked up the Voxtral-patched loader.  We do this
in a subprocess so we get a completely fresh Python interpreter and
don't inherit any `vllm_neuron` module already imported by this
notebook's kernel (which would have been the DLAMI's pre-install).


In [5]:
!python -c "from vllm_neuron.worker.neuronx_distributed_model_loader import NeuronVoxtralForCausalLM; print('NeuronVoxtralForCausalLM present.')"


/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:100: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:102: UserWarning: Warning: Failed to import blockwise_mm_baseline_shard_hidden: No module named 'neuronxcc.nki._private.blockwise_mm'
  warnings.warn(f"Warning: {error}")
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:102: UserWarning: Warning: Failed to import blockwise_mm_bwd: No module named 'neuronxcc.nki._private.blockwise_mm_bwd'
  warnings.warn(f"Warning: {error}")
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:102: UserWarning: Warning: Failed to import blockwise

/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/lib/python3.12/site-packages/neuronx_distributed_inference/utils/constants.py:1: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from neuronx_distributed_inference.models.dbrx.modeling_dbrx import NeuronDbrxForCausalLM


/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/lib/python3.12/site-packages/neuronx_distributed_inference/utils/constants.py:6: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from neuronx_distributed_inference.models.mixtral.modeling_mixtral import NeuronMixtralForCausalLM


/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/lib/python3.12/site-packages/neuronx_distributed_inference/utils/constants.py:14: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from neuronx_distributed_inference.models.qwen3_moe.modeling_qwen3_moe import NeuronQwen3MoeForCausalLM


INFO 07-30 16:09:31 [__init__.py:43] Available plugins for group vllm.platform_plugins:
INFO 07-30 16:09:31 [__init__.py:45] - neuron -> vllm_neuron:register
INFO 07-30 16:09:31 [__init__.py:48] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO 07-30 16:09:31 [__init__.py:212] Platform plugin neuron is activated


INFO 07-30 16:09:32 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 07-30 16:09:32 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.


WARNING 07-30 16:09:32 [interface.py:225] Failed to import from vllm._C: ImportError('libcuda.so.1: cannot open shared object file: No such file or directory')


NeuronVoxtralForCausalLM present.


## 5. Install audio dependencies

`mistral_common[audio]` provides the tokenizer and audio pre-processor
that Voxtral's chat template uses.  `soundfile` and `librosa` are for
the benchmark harness's audio loader.  (Most of these are already in
the DLAMI venv; the cell is a no-op if so.)


In [6]:
!pip install --quiet 'mistral_common[audio]>=1.8.1' 'transformers>=4.54.0' \
    soundfile librosa 2>&1 | tail -3


## 6. Download the Voxtral-Mini-3B checkpoint

Voxtral-Mini-3B is a **gated** repo on HuggingFace.  Before running the
next cell:

1. Visit <https://huggingface.co/mistralai/Voxtral-Mini-3B-2507> and
   accept the license.
2. Generate an HF access token at
   <https://huggingface.co/settings/tokens>.
3. Run `huggingface-cli login` in a terminal and paste your token, OR
   set `HF_TOKEN` in the cell below.

The download is ~10 GB.


In [7]:
MODEL_DIR = HOME / "models" / "Voxtral-Mini-3B-2507"

if not MODEL_DIR.exists():
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="mistralai/Voxtral-Mini-3B-2507",
        local_dir=str(MODEL_DIR),
        # token=os.environ.get("HF_TOKEN"),  # or run `huggingface-cli login`
    )
else:
    print(f"{MODEL_DIR} already exists.")
!ls -lah {MODEL_DIR} | head


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/108 [00:00<?, ?B/s]

...

params.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

consolidated.safetensors:   0%|          | 0.00/9.35G [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

tekken.json:   0%|          | 0.00/14.9M [00:00<?, ?B/s]

total 18G
drwxrwxr-x 3 ubuntu ubuntu 4.0K Jul 30 16:12 .
drwxrwxr-x 3 ubuntu ubuntu 4.0K Jul 30 16:12 ..
drwxrwxr-x 3 ubuntu ubuntu 4.0K Jul 30 16:09 .cache
-rw-rw-r-- 1 ubuntu ubuntu 1.6K Jul 30 16:09 .gitattributes
-rw-rw-r-- 1 ubuntu ubuntu  17K Jul 30 16:09 README.md
-rw-rw-r-- 1 ubuntu ubuntu 1.4K Jul 30 16:09 config.json
-rw-rw-r-- 1 ubuntu ubuntu 8.8G Jul 30 16:09 consolidated.safetensors
-rw-rw-r-- 1 ubuntu ubuntu  108 Jul 30 16:09 generation_config.json
-rw-rw-r-- 1 ubuntu ubuntu 4.7G Jul 30 16:09 model-00001-of-00002.safetensors


## 7. Point vLLM at the NxDI Voxtral contrib

vLLM-neuron's Voxtral loader imports `NeuronApplicationVoxtral` from
the NxDI contrib.  Add the contrib `src/` to `PYTHONPATH` so the
vLLM engine's worker process can import it.


In [8]:
CONTRIB_SRC = NXDI_FORK / "contrib" / "models" / "voxtral-mini-3B" / "src"
sys.path.insert(0, str(CONTRIB_SRC))
os.environ["PYTHONPATH"] = (
    str(CONTRIB_SRC) + os.pathsep + os.environ.get("PYTHONPATH", "")
)
print("Contrib src:", CONTRIB_SRC)

from modeling_voxtral import NeuronApplicationVoxtral  # noqa: F401
print("Import OK.")


Contrib src: /home/ubuntu/neuronx-distributed-inference/contrib/models/voxtral-mini-3B/src


Import OK.


/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:100: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:102: UserWarning: Warning: Failed to import blockwise_mm_baseline_shard_hidden: No module named 'neuronxcc.nki._private.blockwise_mm'
  warnings.warn(f"Warning: {error}")
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:102: UserWarning: Warning: Failed to import blockwise_mm_bwd: No module named 'neuronxcc.nki._private.blockwise_mm_bwd'
  warnings.warn(f"Warning: {error}")
/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:102: UserWarning: Warning: Failed to import blockwise

## 8. Compile the model (one-time, a few minutes)

vLLM will compile Voxtral on first use.  To make the walkthrough
deterministic, we compile ahead of time via
`NeuronApplicationVoxtral.compile()`.  The flags we pick -- TP=4,
`n_positions=768`, `seq_len=768`, on-device sampling, and
`move_trace_to_device` -- match the recommended production
configuration documented in the parent README (§ "Optimizations
shipped in this contrib").

**Important**: `seq_len=768` matches `n_positions=768`.  When serving
via vLLM V1's continuous scheduler this constraint prevents the NxDI
TKG NEFF from being fed positions outside its compiled range during
decode.  Standalone `NeuronApplicationVoxtral.transcribe()` works
with any `seq_len < n_positions`, but vLLM 0.16.0 requires them to
match.

Subsequent notebook runs skip compilation and reload from
`COMPILED_DIR`.


In [9]:
import torch
COMPILED_DIR = HOME / "compiled" / "voxtral_mini_3b_tp4_ods_768_768"
os.environ["NEURON_COMPILED_ARTIFACTS"] = str(COMPILED_DIR)

marker = COMPILED_DIR / "text_decoder" / "text_model" / "model.pt"
if marker.exists():
    print(f"Already compiled at {COMPILED_DIR}.")
else:
    print(f"Compiling to {COMPILED_DIR}. This takes several minutes"
          " the first time.")
    app = NeuronApplicationVoxtral(
        model_path=str(MODEL_DIR),
        tp_degree=4,
        seq_len=768,             # matches n_positions
        n_positions=768,         # KV cache sized for a single 30 s clip
        dtype=torch.bfloat16,
        on_device_sampling=True,     # greedy argmax on-device
        move_trace_to_device=True,   # pre-stage encoder on NeuronCore 0
    )
    app.compile(str(COMPILED_DIR))
    del app
    import gc; gc.collect()
    print("Compile done.")


Compiling to /home/ubuntu/compiled/voxtral_mini_3b_tp4_ods_768_768. This takes several minutes the first time.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

.


Compiler status PASS


...

INFO:Neuron:Done preparing weight layout transformation


INFO:Neuron:Finished building model in 104.2655816078186 seconds


INFO:Neuron:SKIPPING pre-sharding the checkpoints. The checkpoints will be sharded during load time.



Compiler status PASS


Compile done.


## 9. Launch vLLM

vLLM-neuron loads the pre-compiled model from
`NEURON_COMPILED_ARTIFACTS` and serves it via the standard `LLM`
in-process API.  We keep the engine in-process (rather than the OpenAI
HTTP server) so this notebook is self-contained.

**`max_num_seqs=1` is required.**  Batch size > 1 is not supported by
this contrib (see Known Limitations at the end of the notebook).

**`allowed_local_media_path='/'`** lets vLLM read audio files from
anywhere on the filesystem.  Restrict this to your dataset directory
in production.


In [10]:
from vllm import LLM, SamplingParams

llm = LLM(
    model=str(MODEL_DIR),
    tokenizer_mode="mistral",
    tensor_parallel_size=4,      # TP=4 on trn2.3xlarge (LNC=2, four cores)
    dtype="bfloat16",
    max_num_seqs=1,              # BS>1 not supported
    max_model_len=768,           # must match compiled n_positions
    enable_prefix_caching=False,
    allowed_local_media_path="/",
    additional_config={
        "override_neuron_config": {
            "on_device_sampling": True,
            "move_trace_to_device": True,
            "n_positions": 768,
            "seq_len": 768,
        },
    },
)
print("vLLM engine ready.")


INFO 07-30 16:14:21 [__init__.py:43] Available plugins for group vllm.platform_plugins:


INFO 07-30 16:14:21 [__init__.py:45] - neuron -> vllm_neuron:register


INFO 07-30 16:14:21 [__init__.py:48] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.


INFO 07-30 16:14:21 [__init__.py:212] Platform plugin neuron is activated


INFO 07-30 16:14:23 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.


...

(EngineCore_DP0 pid=9089) [2026-07-30 16:14:46] INFO modeling_voxtral.py:686: All model components loaded successfully!
(EngineCore_DP0 pid=9089) [2026-07-30 16:14:46] INFO neuronx_distributed_model_loader.py:967: Voxtral model loaded successfully via NeuronMultiModalCausalLM pattern
(EngineCore_DP0 pid=9089) [2026-07-30 16:14:46] INFO neuronx_distributed_model_runner.py:567: Hardware sampling enabled: config=<neuronx_distributed_inference.models.config.OnDeviceSamplingConfig object at 0x7bc9a4d62d80>


(EngineCore_DP0 pid=9089) INFO 07-30 16:14:46 [kv_cache_utils.py:1307] GPU KV cache size: 1,536 tokens
(EngineCore_DP0 pid=9089) INFO 07-30 16:14:46 [kv_cache_utils.py:1312] Maximum concurrency for 768 tokens per request: 2.00x
(EngineCore_DP0 pid=9089) INFO 07-30 16:14:46 [core.py:278] init engine (profile, create kv cache, warmup model) took 0.00 seconds
(EngineCore_DP0 pid=9089) WARNING 07-30 16:14:46 [scheduler.py:166] Using custom scheduler class vllm_neuron.core.scheduler.ContinuousBatchingNeuronScheduler. This scheduler interface is not public and compatibility may not be maintained.


(EngineCore_DP0 pid=9089) INFO 07-30 16:14:46 [vllm.py:689] Asynchronous scheduling is enabled.
INFO 07-30 16:14:47 [llm.py:355] Supported tasks: ['generate']


vLLM engine ready.


(EngineCore_DP0 pid=9089) [2026-07-30 16:14:46] INFO platform_overrides.py:22: Skipping attention head divisibility check for Neuron platform
(EngineCore_DP0 pid=9089) [2026-07-30 16:14:46] WARNING platform.py:172: Could not import OpenAI serving modules for overrides: No module named 'vllm.entrypoints.openai.serving_engine'
(EngineCore_DP0 pid=9089) [2026-07-30 16:14:46] INFO platform.py:236: Neuron engine client override applied successfully
(EngineCore_DP0 pid=9089) [2026-07-30 16:14:46] INFO platform.py:351: The custom Neuron scheduler will disable chunked prefill and schedule requests using the continuous batching mechanism, prioritizing prefill over decode.
(EngineCore_DP0 pid=9089) [2026-07-30 16:14:46] INFO platform.py:364: Neuron custom scheduler default: max_num_batched_tokens set to 131072. Override with --max-num-batched-tokens if needed.


## 10. Single-file smoke test

Transcribe one 15 s TED-style clip to prove the pipeline works.
vLLM's Mistral chat template accepts audio inputs via
`{"type": "audio_url", "audio_url": {"url": <path>}}`.

Provide any 16 kHz mono `.wav` file (or a URL that `mistral_common`
can download).  The cell below fetches a public TED sample and
trims/downsamples it to 15 s mono 16 kHz.


In [11]:
import urllib.request
AUDIO_URL = (
    "https://huggingface.co/datasets/reach-vb/random-audios/"
    "resolve/main/ted_60.wav"
)
RAW_AUDIO = HOME / "sample_audio" / "ted_60.wav"
SAMPLE_AUDIO = HOME / "sample_audio" / "ted_15_mono16k.wav"
RAW_AUDIO.parent.mkdir(parents=True, exist_ok=True)

if not RAW_AUDIO.exists():
    urllib.request.urlretrieve(AUDIO_URL, RAW_AUDIO)

if not SAMPLE_AUDIO.exists():
    import soundfile as sf, librosa, numpy as np
    data, sr = sf.read(str(RAW_AUDIO))
    if data.ndim == 2:
        data = data.mean(axis=1)          # stereo -> mono
    data = librosa.resample(data.astype(np.float32),
                            orig_sr=sr, target_sr=16000)
    data = data[:15 * 16000]              # trim to 15 s
    sf.write(str(SAMPLE_AUDIO), data, 16000)

print("Audio file:", SAMPLE_AUDIO,
      SAMPLE_AUDIO.stat().st_size, "bytes")


Audio file: /home/ubuntu/sample_audio/ted_15_mono16k.wav 480044 bytes


In [12]:
conversation = [{
    "role": "user",
    "content": [
        {"type": "audio_url",
         "audio_url": {"url": f"file://{SAMPLE_AUDIO}"}},
        {"type": "text", "text": "Transcribe this audio."},
    ],
}]

sampling = SamplingParams(temperature=0.0, max_tokens=256)

import time
t0 = time.perf_counter()
outputs = llm.chat(conversation, sampling_params=sampling)
elapsed = time.perf_counter() - t0

text = outputs[0].outputs[0].text
print(f"Latency: {elapsed * 1000:.1f} ms")
print(f"Tokens:  {len(outputs[0].outputs[0].token_ids)}")
print(f"\nTranscription:\n{text}")


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

/opt/aws_neuronx_venv_pytorch_inference_vllm_0_16/lib/python3.12/site-packages/mistral_common/tokens/tokenizers/tekken.py:471: FutureWarning: `get_control_token` is deprecated. Use `get_special_token` instead.
  warnings.warn("`get_control_token` is deprecated. Use `get_special_token` instead.", FutureWarning)


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Latency: 654.0 ms
Tokens:  48

Transcription:
So in college, I was a government major, which means I had to write a lot of papers. Now, when a normal student writes a paper, they might spread the work out a little like this. So, you know.


## 11. Run the benchmark harness

The `benchmark_harness/` directory alongside this notebook has a
serial driver that measures latency per audio file.  To run it in
production:

1. Populate `benchmark_harness/dataset/` with your own audio files
   (16 kHz mono WAV recommended).
2. Add one row per file to `benchmark_harness/dataset/manifest.csv`
   in the `audio_path,duration_sec,transcript` format.
3. Run the cell below.

For this walkthrough we auto-generate a 5-clip mix from the TED
sample downloaded in cell 22 (durations 5, 10, 15, 20, 25 seconds)
so the harness has something to measure end-to-end.  You would
replace this cell with your own dataset population step.

**Reference measurement** from a prior execution: this exact 5-clip
mix, TP=4 LNC=2 BF16 SDK 2.30, `max_new_tokens=256`:

| Duration | Mean per file (ms) |
|:--------:|:------------------:|
| 5-10 s   | ~291               |
| 10-15 s  | ~408               |
| 15-20 s  | ~454               |
| 20-25 s  | ~626               |
| 25-30 s  | ~733               |
| Overall  | **~489 ms mean, 438 ms median** |

Numbers will vary with
content complexity (dense speech → more generated tokens → longer
per-file latency).


In [13]:
# Auto-populate the harness dataset with 5 clips trimmed from the TED
# sample downloaded above.  In production, you'd replace this cell
# with your own manifest + audio files.
HARNESS = NXDI_FORK / "contrib" / "models" / "voxtral-mini-3B" / \
          "benchmark_harness"
MANIFEST = HARNESS / "dataset" / "manifest.csv"
DATASET_DIR = HARNESS / "dataset"

import soundfile as sf
import librosa
import numpy as np

# Load the 60 s TED sample and downsample once to 16 kHz mono.
raw_data, raw_sr = sf.read(str(RAW_AUDIO))
if raw_data.ndim == 2:
    raw_data = raw_data.mean(axis=1)
mono16k = librosa.resample(raw_data.astype(np.float32),
                           orig_sr=raw_sr, target_sr=16000)

# Emit 5, 10, 15, 20, 25 s clips.
durations = [5, 10, 15, 20, 25]
for d in durations:
    out = DATASET_DIR / f"ted_{d}s.wav"
    sf.write(str(out), mono16k[: d * 16000], 16000)

# Rewrite the manifest.
with MANIFEST.open("w") as f:
    f.write("audio_path,duration_sec,transcript\n")
    for d in durations:
        f.write(f"ted_{d}s.wav,{d}.0,\n")

# Recount.
n_manifest = sum(1 for _ in MANIFEST.open()) - 1
print(f"Manifest at {MANIFEST}: {n_manifest} audio rows.")


Manifest at /home/ubuntu/neuronx-distributed-inference/contrib/models/voxtral-mini-3B/benchmark_harness/dataset/manifest.csv: 5 audio rows.


In [14]:
import subprocess

# Free the in-process vLLM engine before the harness runs -- the
# harness will re-create it internally and only one process at a
# time can hold the Neuron cores.
try:
    del llm
except NameError:
    pass
import gc; gc.collect()

cmd = [
    "python", str(HARNESS / "run_voxtral_benchmark.py"),
    "--backend", "vllm_neuron",
    "--manifest", str(MANIFEST),
    "--model-dir", str(MODEL_DIR),
    "--compiled-dir", str(COMPILED_DIR),
    "--tp-degree", "4",
    "--seq-len", "768",
    "--n-positions", "768",
    "--ods",
    "--output-dir", str(HARNESS / "results"),
    "--runs", "3",
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=False)
print("Exit code:", result.returncode)


Running: python /home/ubuntu/neuronx-distributed-inference/contrib/models/voxtral-mini-3B/benchmark_harness/run_voxtral_benchmark.py --backend vllm_neuron --manifest /home/ubuntu/neuronx-distributed-inference/contrib/models/voxtral-mini-3B/benchmark_harness/dataset/manifest.csv --model-dir /home/ubuntu/models/Voxtral-Mini-3B-2507 --compiled-dir /home/ubuntu/compiled/voxtral_mini_3b_tp4_ods_768_768 --tp-degree 4 --seq-len 768 --n-positions 768 --ods --output-dir /home/ubuntu/neuronx-distributed-inference/contrib/models/voxtral-mini-3B/benchmark_harness/results --runs 3


Loading backend 'vllm_neuron'...
INFO 07-30 16:14:54 [__init__.py:43] Available plugins for group vllm.platform_plugins:
INFO 07-30 16:14:54 [__init__.py:45] - neuron -> vllm_neuron:register
INFO 07-30 16:14:54 [__init__.py:48] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO 07-30 16:14:54 [__init__.py:212] Platform plugin neuron is activated


INFO 07-30 16:14:57 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 07-30 16:14:57 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.


INFO 07-30 16:14:58 [utils.py:223] non-default args: {'tokenizer_mode': 'mistral', 'allowed_local_media_path': '/', 'dtype': 'bfloat16', 'max_model_len': 768, 'tensor_parallel_size': 4, 'enable_prefix_caching': False, 'max_num_seqs': 1, 'disable_log_stats': True, 'additional_config': {'override_neuron_config': {'on_device_sampling': True, 'move_trace_to_device': True, 'n_positions': 768, 'seq_len': 768}}, 'model': '/home/ubuntu/models/Voxtral-Mini-3B-2507'}
INFO 07-30 16:14:58 [model.py:529] Resolved architecture: VoxtralForConditionalGeneration
INFO 07-30 16:14:58 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=768.
INFO 07-30 16:14:58 [vllm.py:689] Asynchronous scheduling is enabled.
WARNING 07-30 16:14:58 [interface.py:225] Failed to import from vllm._C: ImportError('libcuda.so.1: cannot open shared object file: No such file or directory')


[2026-07-30 16:14:58] INFO platform.py:111: Applying Neuron config overrides
[2026-07-30 16:14:58] INFO platform.py:127: Neuron config overrides applied successfully
[2026-07-30 16:14:58] INFO platform_overrides.py:22: Skipping attention head divisibility check for Neuron platform
[2026-07-30 16:14:58] WARNING platform.py:172: Could not import OpenAI serving modules for overrides: No module named 'vllm.entrypoints.openai.serving_engine'
[2026-07-30 16:14:58] INFO platform.py:236: Neuron engine client override applied successfully
[2026-07-30 16:14:58] INFO platform.py:351: The custom Neuron scheduler will disable chunked prefill and schedule requests using the continuous batching mechanism, prioritizing prefill over decode.
[2026-07-30 16:14:58] INFO platform.py:364: Neuron custom scheduler default: max_num_batched_tokens set to 131072. Override with --max-num-batched-tokens if needed.


...

  [4/5] ted_20s.wav:  628.4 ms


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s, est. speed input: 528.37 toks/s, output: 122.46 toks/s]


  [5/5] ted_25s.wav:  732.0 ms


  Mean latency: 484.4 ms

All runs mean-of-means: 501.9 ms/file


Exit code: 0


In [15]:
!python {HARNESS}/summarize_results.py {HARNESS}/results/


N files:        15
Mean latency:   501.9 ms
Median latency: 458.9 ms
P90 latency:    733.4 ms
P99 latency:    745.4 ms
Stdev:          176.0 ms

Per-duration bin (mean latency):
  bin          n    mean ms
  5-10 s       3      291.6
  10-15 s      3      404.8
  15-20 s      3      453.1
  20-25 s      3      621.9
  25-30 s      3      737.9


## 12. Known Limitations

- **Batch size 1 only.**  The stock `ImageToTextModelWrapper` uses
  `scatter_by_index_put` in a way that is not shape-safe for batch
  size > 1 context encoding.  `max_num_seqs=1` is set above.
  Serving concurrent requests requires multiple engines (one per
  NeuronCore group) or an unmerged scatter fix.
- **30 s audio maximum per request.**  Voxtral upstream supports 30
  min transcribe / 40 min understand modes; those code paths have not
  been validated on Neuron.  Clips longer than 30 s are truncated by
  the audio encoder.
- **`seq_len` must match `n_positions` for vLLM serving.**  Standalone
  `NeuronApplicationVoxtral.transcribe()` works with `seq_len` shorter
  than `n_positions` (e.g. 512 / 768) which is ~5% faster on SDK 2.30.
  vLLM 0.16.0's V1 scheduler can feed the TKG NEFF positions between
  `seq_len` and `n_positions`, which the compiled NEFF rejects with
  NRT status 1006.  Keep `seq_len == n_positions` for vLLM.
- **Continuous batching and streaming responses** (`stream=true` on
  the HTTP API) have not been benchmarked on this configuration.
- **Function calling** (Voxtral-Small-24B only) is out of scope for
  this notebook.
- **Voxtral-Small-24B** is not covered; it uses the same architecture
  and could be onboarded at TP=4 with the same
  `NeuronApplicationVoxtral` pattern but has not been validated in
  this branch.

## Where to file issues

- vLLM-neuron Voxtral loader:
  <https://github.com/jimburtoft/vllm-neuron/tree/contrib/voxtral-0.5.0>
- NxDI Voxtral contrib:
  <https://github.com/jimburtoft/neuronx-distributed-inference/tree/contrib/voxtral-mini-3B>

After the upstream PR merges, the branches above will be replaced by
`aws-neuron/neuronx-distributed-inference` and
`vllm-project/vllm-neuron` releases.
